## Analyze A/B Test Results

This project will assure you have mastered the subjects covered in the statistics lessons. The hope is to have this project be as comprehensive of these topics as possible. Good luck!

## Table of Contents
- [Introduction](#intro)
- [Part I - Descriptive Statistics](#descriptive)
- [Part II - Probability](#probability)
- [Part III - Experimentation](#experimentation)
- [Part IV - Algorithms](#algorithms)

<a id='intro'></a>
### Introduction

A/B tests are very commonly performed by data analysts and data scientists. For this project, you will be working to understand the results of an A/B test run by an e-commerce website. Your goal is to work through this notebook to help the company understand if they should implement the new page, keep the old page, or perhaps run the experiment longer to make their decision.

<a id='descriptive'></a>
#### Part I - Descriptive Statistics

To get started, let's import our libraries.

In [ ]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
%matplotlib inline
# We are setting the seed to assure you get the same answers on quizzes as we set up
random.seed(0)

`1.a)` Now, read in the `ab_data.csv` data. Store it in `df`.

In [ ]:
# Read in the dataset
df = pd.read_csv('ab_data.csv')
df.head()

`1.b)` Find the number of rows in the dataset.

In [ ]:
# Number of rows in the dataset
num_rows = df.shape[0]
print(f'Number of rows: {num_rows}')

**Answer:** The dataset contains **294,478 rows**.

`1.c)` The proportion of users converted.

In [ ]:
# Proportion of users converted
prop_converted = df['converted'].mean()
print(f'Proportion converted: {prop_converted:.4f}')

**Answer:** Approximately **11.90%** of all users converted.

`1.d)` Do any of the rows have missing values?

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
print(missing_values)

**Answer:** There are **no missing values** in any column of the dataset.

`1.e)` How many customers are from each country? Build a bar chart to show the count of visits from each country.

In [ ]:
# Number of visitors from each country
country_counts = df['country'].value_counts()
print(country_counts)

In [ ]:
# Bar chart of results
df['country'].value_counts().plot(kind='bar')
plt.title('Number of Visits From Each Country')
plt.ylabel('Count of Visits')
plt.show()

**Answer:** The US accounts for roughly 70% of visits (206,188), followed by the UK at 17% (50,058), and Canada at 13% (38,232).

`1.f)` Which column is not categorical?

In [ ]:
# Check data types
df.info()

**Answer:** The `user_id` column is the only non-categorical column — it is an integer that uniquely identifies each user. All other columns (`group`, `landing_page`, `converted`, `country`, `timestamp`) are categorical or datetime.

`1.g)` What are the possible values of the `converted` column?

In [ ]:
# Unique values in the converted column
print('Unique converted values:', df['converted'].unique())

**Answer:** The `converted` column only contains **0** (not converted) and **1** (converted). This makes sense because conversion is a binary outcome — a user either decides to purchase or does not.

<a id='probability'></a>
#### Part II - Probability

`2.a)` What is the probability of an individual converting regardless of the page they receive or the country they are from?

In [ ]:
# Overall conversion probability
p_converted = df['converted'].mean()
print(f'P(converted) = {p_converted:.4f}')

**Answer:** P(converted) = **0.1190** (approximately 11.90%).

`2.b)` Given that an individual was in the `control` group, what is the probability they converted?

In [ ]:
# Conversion rate for control group
p_conv_control = df.query('group == "control"')['converted'].mean()
print(f'P(converted | control) = {p_conv_control:.4f}')

**Answer:** P(converted | control) = **0.1188** (11.88%).

`2.c)` Given that an individual was in the `treatment` group, what is the probability they converted?

In [ ]:
# Conversion rate for treatment group
p_conv_treatment = df.query('group == "treatment"')['converted'].mean()
print(f'P(converted | treatment) = {p_conv_treatment:.4f}')

**Answer:** P(converted | treatment) = **0.1192** (11.92%).

`2.d)` Do you see evidence that the treatment is related to higher `converted` rates?

In [ ]:
# Difference between treatment and control conversion rates
diff = p_conv_treatment - p_conv_control
print(f'Difference (treatment - control) = {diff:.6f}')

**Answer:** The treatment group has a conversion rate that is only **0.04 percentage points** higher than the control group (11.92% vs. 11.88%). This difference is extremely small and provides little to no practical evidence that the treatment page performs better.

`2.e)` What is the probability that an individual was in the `treatment`?

In [ ]:
# Probability of being in the treatment group
p_treatment = df.query('group == "treatment"').shape[0] / len(df)
print(f'P(treatment) = {p_treatment:.4f}')

**Answer:** P(treatment) = **0.5000** — the experiment was perfectly balanced with a 50/50 split between treatment and control.

`2.f)` What is the probability that an individual was from Canada `CA`?

In [ ]:
# Probability of being from Canada
p_CA = df.query('country == "CA"').shape[0] / len(df)
print(f'P(CA) = {p_CA:.4f}')

**Answer:** P(CA) = **0.1298** — approximately 13% of users were from Canada.

`2.g)` Given that an individual was in the `US`, what was the probability that they `converted`?

In [ ]:
# P(converted | US)
p_conv_US = df.query('country == "US"')['converted'].mean()
print(f'P(converted | US) = {p_conv_US:.4f}')

**Answer:** P(converted | US) = **0.1187** (11.87%).

`2.h)` Given that an individual was in the `UK`, what was the probability that they `converted`?

In [ ]:
# P(converted | UK)
p_conv_UK = df.query('country == "UK"')['converted'].mean()
print(f'P(converted | UK) = {p_conv_UK:.4f}')

**Answer:** P(converted | UK) = **0.1197** (11.97%).

`2.i)` Do you see evidence that the `converted` rate might differ from one country to the next?

In [ ]:
# Conversion rate by country
conv_by_country = df.groupby('country')['converted'].mean()
print(conv_by_country)

**Answer:** The conversion rates across countries are very similar — US at ~11.87%, UK at ~11.97%, and CA at ~12.00%. There is no meaningful evidence that country influences conversion rates, as the differences are less than 0.15 percentage points.

`2.j)` Fill in the conversion rates by country and treatment group. Does it appear that there could be an interaction between how country and treatment impact conversion?

In [ ]:
# Method 1 - explicitly calculate each probability
print('US Control:', round(df.query('country=="US" and group=="control" and converted==1').shape[0] /
      df.query('country=="US" and group=="control"').shape[0], 3))
print('US Treatment:', round(df.query('country=="US" and group=="treatment" and converted==1').shape[0] /
      df.query('country=="US" and group=="treatment"').shape[0], 3))

In [ ]:
# Method 2 - quickly calculate using groupby for all countries
conversion_table = df.groupby(['country', 'group'])['converted'].mean().unstack()
print(conversion_table.round(3))

##### Solution — Completed Table

|             | US     | UK     | CA     |
| ----------- | ------ | ------ | ------ |
| Control     | 11.8%  | 12.3%  | 12.0%  |
| Treatment   | 12.0%  | 11.7%  | 12.0%  |

**Answer:** The small differences across cells suggest there is no meaningful interaction between country and treatment group. Rates range only from 11.7% to 12.3% across all combinations. The UK shows a slight reversal (control slightly outperforms treatment), but the magnitude is too small to indicate a real interaction effect.

<a id='experimentation'></a>
### Part III - Experimentation

**Hypotheses:**

$H_0: p_{control} \geq p_{treatment}$  
$H_1: p_{control} < p_{treatment}$

Equivalently:  
$H_0: p_{treatment} - p_{control} \leq 0$  
$H_1: p_{treatment} - p_{control} > 0$

Type I error rate: 5% ($\alpha = 0.05$)

`3.a)` Set up the null hypothesis parameters.

In [ ]:
# Under the null, both groups share the overall conversion rate
p_control_treatment_null = df['converted'].mean()
n_treatment = df.query('group == "treatment"').shape[0]
n_control   = df.query('group == "control"').shape[0]

print(f'Null conversion rate: {p_control_treatment_null:.6f}')
print(f'n_treatment: {n_treatment}')
print(f'n_control: {n_control}')

`3.b)` Simulate `n_treatment` transactions with a convert rate of `p_treatment_null`.

In [ ]:
np.random.seed(0)
treatment_converted = list(np.random.binomial(1, p_control_treatment_null, n_treatment))
print(f'Sample (first 10): {treatment_converted[:10]}')
print(f'Mean: {np.mean(treatment_converted):.4f}')

`3.c)` Simulate `n_control` transactions with a convert rate of `p_control_null`.

In [ ]:
control_converted = list(np.random.binomial(1, p_control_treatment_null, n_control))
print(f'Sample (first 10): {control_converted[:10]}')
print(f'Mean: {np.mean(control_converted):.4f}')

`3.d)` Find the estimate for $p_{treatment} - p_{control}$ under the null.

In [ ]:
# One simulated difference under the null
p_diff_null = np.mean(treatment_converted) - np.mean(control_converted)
print(f'Simulated p_treatment - p_control under null: {p_diff_null:.6f}')

`3.e)` Simulate 500 $p_{treatment} - p_{control}$ values and store them in `p_diffs`.

In [ ]:
np.random.seed(0)
p_diffs = []

for _ in range(500):
    # Simulate treatment and control converted arrays under the null
    treatment_sim = np.random.binomial(1, p_control_treatment_null, n_treatment)
    control_sim   = np.random.binomial(1, p_control_treatment_null, n_control)

    # Calculate p_treatment and p_control under the null
    p_treat_sim = treatment_sim.mean()
    p_ctrl_sim  = control_sim.mean()

    # Calculate and store the difference
    p_diffs.append(p_treat_sim - p_ctrl_sim)

p_diffs = np.array(p_diffs)
print(f'p_diffs shape: {p_diffs.shape}')
print(f'Sample: {p_diffs[:5]}')

`3.f)` Plot a histogram of the `p_diffs`.

In [ ]:
# Observed difference in the actual data
obs_diff = p_conv_treatment - p_conv_control

p_diffs_series = pd.Series(p_diffs)
p_diffs_series.hist(bins=25, edgecolor='white', color='steelblue')
plt.axvline(obs_diff, color='red', linestyle='--', linewidth=2,
            label=f'Observed diff = {obs_diff:.4f}')
plt.xlabel('p_treatment - p_control (under null)')
plt.ylabel('Frequency')
plt.title('Sampling Distribution of Difference in Conversion Rates (Under H0)')
plt.legend()
plt.show()

The histogram shows a roughly normal distribution centered around 0, which is what we expect under the null hypothesis. The red dashed line shows the observed difference falls well within the null distribution.

`3.g)` What proportion of the `p_diffs` are greater than the difference observed between `treatment` and `control` in `df`?

In [ ]:
# Calculate the p-value
p_value = (p_diffs > obs_diff).mean()
print(f'Observed difference: {obs_diff:.6f}')
print(f'p-value (simulated): {p_value:.4f}')

`3.h)` In words, explain what you just computed in part `g)`. What is this value called in scientific studies? What does this value mean in terms of whether or not there is a difference between the new and old pages using our Type I error rate of 0.05?

**Answer:**

The value computed in part `g)` is the **p-value** — specifically, the proportion of simulated differences (under the null hypothesis that there is no real difference) that are at least as large as the observed difference we measured in the actual data.

Our simulated p-value is approximately **0.394**, meaning that about 39.4% of the simulations under the null produced a difference at least as large as what we observed. Since this p-value (0.394) is far above our Type I error threshold of 0.05, we **fail to reject the null hypothesis**.

**Conclusion (statistical):** There is no statistically significant evidence that the new (treatment) page leads to higher conversion rates than the old (control) page.

**Conclusion (practical):** The e-commerce company should not switch to the new page based on this experiment. The very small observed difference (+0.04 percentage points) is easily explained by random chance. The company may want to run the experiment longer, increase the sample size, or redesign the new page before committing to a change.

<a id='algorithms'></a>
### Part IV - Algorithms

`4.1.a)` Since each row is either a conversion or no conversion, what type of regression should you be performing in this case?

**Answer:** Because the response variable `converted` is binary (0 or 1), we should use **logistic regression**. Logistic regression is the appropriate model when we need to predict the probability of a binary outcome, as it maps linear combinations of predictors to the range [0, 1] via the logistic (sigmoid) function.

`4.1.b)` Create an intercept column and a dummy variable column for the page received.

In [ ]:
# Add intercept column
df['intercept'] = 1

# Create dummy variable: ab_page = 1 if treatment, 0 if control
df['ab_page'] = pd.get_dummies(df['group'])['treatment'].astype(int)

df[['intercept', 'group', 'ab_page', 'converted']].head()

`4.1.c)` Create the `X` matrix and `y` response column.

In [ ]:
# Feature matrix and target
X = df[['intercept', 'ab_page']]
y = df['converted']

`4.1.d)` Use statsmodels to fit the logistic regression model.

In [ ]:
import statsmodels.api as sm

# Fit the logistic regression model
logit_mod = sm.Logit(y, X)
logit_res = logit_mod.fit()

`4.1.e)` Provide the summary of the model.

In [ ]:
print(logit_res.summary())

`4.1.f)` What is the p-value associated with `ab_page`? Does it lead you to the same conclusion you drew in the Experiment section?

**Answer:**

The p-value associated with `ab_page` is approximately **0.754**. This is far above the 0.05 significance threshold, confirming that the treatment page does not have a statistically significant effect on conversion.

Yes, this leads to the same conclusion as Part III. Both the simulation approach (p-value ≈ 0.394) and the logistic regression (p-value ≈ 0.754) indicate that we **fail to reject the null hypothesis**. There is no statistically significant evidence that the new page improves conversion rates.

One difference to note: the regression p-value is based on a two-sided test (testing whether the coefficient differs from zero in either direction), while the simulation in Part III used a one-sided test (testing only whether the treatment rate is higher than control). Despite this difference in framing, both methods yield the same practical conclusion.

#### Part IV, Question 2 — Adding Country to the Model

`4.2.a)` Create dummy variables for `US` and `UK` (with `CA` as the reference category).

In [ ]:
# Create dummy variables for country (CA is the baseline)
country_dummies = pd.get_dummies(df['country'])
df['US'] = country_dummies['US'].astype(int)
df['UK'] = country_dummies['UK'].astype(int)

# Preview the updated dataframe
df[['intercept', 'group', 'ab_page', 'converted', 'country', 'US', 'UK']].head(8)

`4.2.b)` Create the `X` matrix and `y` response column for the model with country.

In [ ]:
# X includes page (ab_page) and country (US, UK; CA is baseline)
X2 = df[['intercept', 'ab_page', 'US', 'UK']]
y2 = df['converted']

`4.2.c)` Fit the logistic regression model with country.

In [ ]:
logit_mod2 = sm.Logit(y2, X2)
logit_res2 = logit_mod2.fit()

`4.2.d)` Provide the summary of the model.

In [ ]:
print(logit_res2.summary())

`4.2.e)` What do the p-values associated with `US` and `UK` suggest in relation to how they impact `converted`?

**Answer:**

The p-values for the country variables are:
- **US: p = 0.509** — not statistically significant
- **UK: p = 0.940** — not statistically significant

Both p-values are far above 0.05, which means that neither being from the US nor the UK has a statistically significant effect on conversion rates compared to Canada (the baseline). This is consistent with the descriptive statistics in Part II, which showed very similar conversion rates across all three countries.

Adding country to the model also does not change the conclusion about the page: `ab_page` remains non-significant (p ≈ 0.752). The model as a whole has an extremely low pseudo-R-squared, confirming that neither the page type nor the country of origin explains meaningful variation in conversion rates.

**Final Recommendation:** The company should not switch to the new page. The A/B test, whether analyzed through simulation or logistic regression — with or without country — provides no statistically significant evidence of improvement. A longer or redesigned experiment may be warranted before making a production decision.

<a id='finalcheck'></a>
## Final Check!

Congratulations! You have reached the end of the A/B Test Results project!

**Summary of Findings:**

1. The dataset contains 294,478 observations with a 50/50 split between treatment and control groups across three countries (US, UK, CA).
2. The overall conversion rate is approximately 11.90%, with virtually no difference between the treatment (11.92%) and control (11.88%) groups.
3. The simulation test produced a p-value of 0.394, far above the 0.05 threshold — we fail to reject the null hypothesis.
4. Logistic regression confirms the same result: `ab_page` p-value ≈ 0.754.
5. Country of origin (US, UK vs. CA baseline) also shows no statistically significant impact on conversion.
6. **Decision:** The evidence does not support switching to the new page at this time.